# 环节 02 · 值方法与 DQN（配套 Notebook）

> 配套长文：[环节02-值方法与DQN详解.md](./环节02-值方法与DQN详解.md)
> 定位：把 DP / MC / TD / Q-learning 在同一套网格世界上跑出来，**并用长文 §8 的真实表格做交叉验证**。全部**纯 Python 标准库**。

**怎么跑**

- 依赖：无。逐格 `Shift+Enter`。
- 改 `EPISODES` 看探索覆盖怎么随回合数变化；改 `random_start` 看"没访问过就学不到"。

**地图：本 Notebook ↔ 长文章节**

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 网格世界 | §8 | step / 奖励落在"转移进入"这一点 |
| §2 值迭代真值 | §2 / §8 | 有模型时的 V*（0.656 … 1.000） |
| §3 MC vs TD(0) | §3 / §4 | 一条轨迹上两种更新的传播方式 |
| §4 n-step | §4 | n 越大越接近 MC |
| §5 Q-learning vs SARSA | §5 | 最大化的偏置从哪来 |
| §6 探索覆盖 | §8 | 固定起点 ⟹ (3,0) 访问 0 次、完全没学到 |
| §7 贪心策略 | §8 | 漏学的状态给出错误策略（最大偏差 0.729） |


## 1. 网格世界：4 宽 × 3 高

- 起点 `(0,0)`；转移到目标 `(3,2)` → **+1**；转移到陷阱 `(3,1)` → **−1**；其他转移 0；撞墙留在原地；γ = 0.9。
- 注意：奖励是在**转移进入**目标/陷阱时给的（不是"停在终止状态"）。


In [ ]:
W, H, GOAL, PIT = 4, 3, (3, 2), (3, 1)
TERM = {GOAL, PIT}
GAMMA = 0.9
ACTS = ((0, 1), (0, -1), (-1, 0), (1, 0))          # 上 下 左 右
STATES = [(x, y) for x in range(W) for y in range(H)]

def step(s, a):
    ns = (s[0] + a[0], s[1] + a[1])
    if not (0 <= ns[0] < W and 0 <= ns[1] < H):
        ns = s                                     # 撞墙：留在原地
    if ns == GOAL:
        return ns, 1.0, True                       # 转移到目标 → +1
    if ns == PIT:
        return ns, -1.0, True                      # 转移到陷阱 → −1
    return ns, 0.0, False

def show(V, title):
    print(title)
    for y in range(H - 1, -1, -1):
        row = []
        for x in range(W):
            if (x, y) == GOAL:
                row.append("      G")
            elif (x, y) == PIT:
                row.append("      P")
            else:
                row.append(f"{V[(x, y)]:7.3f}")
        print("  ".join(row))
    print()

print("测试一步：从 (2,2) 往右 →", step((2, 2), (1, 0)))
print("测试撞墙：从 (0,0) 往左 →", step((0, 0), (-1, 0)))


## 2. 值迭代：有模型时的真值 V\*

长文 §8 的表：起点 `0.656 = 0.9⁴`（奖励延迟 5 步 → 折扣 γ⁴），紧挨目标那格 `1.000`（奖励第 1 步到，γ⁰）。


In [ ]:
V = {s: 0.0 for s in STATES}
for _ in range(500):
    V = {s: (0.0 if s in TERM else
             max(r + (0.0 if d else GAMMA * V[ns])
                 for (ns, r, d) in (step(s, a) for a in ACTS)))
         for s in STATES}
show(V, "真值（值迭代，有模型）：")
print("读法：0.9⁴ =", round(0.9**4, 4), " ← 起点到目标要 5 个动作，奖励晚到 5 步")


## 3. MC vs TD(0)：同一条轨迹，两种更新

**MC**：跑完整条轨迹，用**真实回报** `G_t` 更新（等得久、无偏、方差大）。
**TD(0)**：用 `r + γV(s')`（自举）边走边更新（有偏、方差小，且**每步只把信息往回传一格**）。

用一条安全路径 `(0,0)→(0,1)→(0,2)→(1,2)→(2,2)→目标`，每步奖励 0、最后一步 +1。


In [ ]:
TRAJ = [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (3, 2)]
rewards = [0.0, 0.0, 0.0, 0.0, 1.0]                # 每次转移的奖励（最后一步进目标 = +1）
ALPHA = 0.5

# —— MC：先算真实回报，再对轨迹上每个状态更新 ——
V_mc = dict.fromkeys(STATES, 0.0)
G = 0.0
targets = []
for r in reversed(rewards):
    G = r + GAMMA * G
    targets.append(G)
targets.reverse()
print("MC：用真实回报 G_t 更新（等整条跑完）")
for s, g in zip(TRAJ[:-1], targets):
    V_mc[s] += ALPHA * (g - V_mc[s])
    print(f"  {s} → G = {g:.4f}   V = {V_mc[s]:.4f}")
print()

# —— TD(0)：边走边用 r + γV(s') 更新 ——
V_td = dict.fromkeys(STATES, 0.0)
print("TD(0)：用 r + γV(s') 更新（自举）")
for k, (s, r) in enumerate(zip(TRAJ[:-1], rewards)):
    s2, done = TRAJ[k + 1], TRAJ[k + 1] in TERM
    target = r + (0.0 if done else GAMMA * V_td[s2])
    V_td[s] += ALPHA * (target - V_td[s])
    print(f"  {s}→{s2}  target = {target:.4f}   V({s}) = {V_td[s]:.4f}")
print("\n→ MC 一遍就把整条路径的价值铺开；TD(0) 一遍只更新了最后一步，(2,2) = 0.5。")


## 4. n-step：介于 TD(0) 与 MC 之间的旋钮

`G^(n)_t = r_{t+1} + γr_{t+2} + … + γ^{n-1}r_{t+n} (+ γ^n V(s_{t+n}))`
n=1 退化成 TD(0)，n→∞ 退化成 MC。回报的"延迟"决定它要打折多少。


In [ ]:
print("从起点 (0,0) 出发的 n 步真实回报（只改 n）：")
for n in (1, 2, 3, 4, 5):
    g = sum((GAMMA ** k) * rewards[k] for k in range(min(n, len(rewards))))
    print(f"  n={n}: G^(n) = {g:.4f}")
print("  n→∞: 0.6561 = 0.9⁴  —— 折扣是「奖励延迟」的代价，不是「走了几步」。")


## 5. Q-learning vs SARSA：一个 max 的差别

- **Q-learning**（off-policy）：`target = r + γ·max_a' Q(s', a')` —— 学"最优策略"。
- **SARSA**（on-policy）：`target = r + γ·Q(s', a')`，a' 是**行为策略实际会选的下一步动作** —— 学"当前在探索的策略"。


In [ ]:
# 手设一个小 Q 表：站在 (2,1)，四个动作的估计值不同
Q = {((2, 1), (0, 1)): 0.8, ((2, 1), (0, -1)): 0.3,
     ((2, 1), (-1, 0)): 0.2, ((2, 1), (1, 0)): 0.1}
s, a = (2, 0), (0, 1)                              # 从 (2,0) 往上走到 (2,1)
s2, r, done = step(s, a)
next_taken = (-1, 0)                               # 假设 ε-greedy 下一步实际选了"左"

q_learn = r + GAMMA * max(Q[(s2, a2)] for a2 in ACTS)
sarsa = r + GAMMA * Q[(s2, next_taken)]
print(f"Q-learning 目标 = r + γ·max_a' Q(s',a') = {q_learn:.3f}   （乐观，学最优策略）")
print(f"SARSA      目标 = r + γ·Q(s', 实际动作)  = {sarsa:.3f}   （如实，学行为策略）")
print("→ Q-learning 的 max 会系统性挑被高估的动作，这就是「最大化偏差」。")


## 6. 探索覆盖：没访问过 = 没学到（长文最想让你记住的一课）

同一套超参，只改**起点是否随机**，结果天差地别。


In [ ]:
import random

ALPHA_Q, EPS, EPISODES = 0.4, 0.1, 20000

def q_learn(random_start, seed=0):
    random.seed(seed)
    Q = {(s, a): 0.0 for s in STATES for a in ACTS}
    visit = dict.fromkeys(STATES, 0)
    for _ in range(EPISODES):
        s = random.choice(STATES) if random_start else (0, 0)
        while s not in TERM:
            visit[s] += 1
            a = (random.choice(ACTS) if random.random() < EPS
                 else max(ACTS, key=lambda a: Q[(s, a)]))
            ns, r, done = step(s, a)
            target = r + (0.0 if done else GAMMA * max(Q[(ns, a2)] for a2 in ACTS))
            Q[(s, a)] += ALPHA_Q * (target - Q[(s, a)])
            s = ns
    Vq = {s: (0.0 if s in TERM else max(Q[(s, a)] for a in ACTS)) for s in STATES}
    return Vq, visit, Q

Vq, visit, Q = q_learn(random_start=False)
show(Vq, "① Q-learning（固定起点 (0,0)，20000 回合）：")
leaked = [s for s in STATES if s not in TERM and visit[s] == 0]
print("访问 0 次的状态：", leaked)
print("访问次数：", {s: visit[s] for s in STATES if s not in TERM})
print("最大偏差：", round(max(abs(Vq[s] - V[s]) for s in STATES if s not in TERM), 3))


In [ ]:
Vq2, _, _ = q_learn(random_start=True)
show(Vq2, "② 只把起点改成随机（其余参数不动）：")
print("最大偏差：", round(max(abs(Vq2[s] - V[s]) for s in STATES if s not in TERM), 3),
      " ← 与真值逐格一致")
print("→ 算法没错，是数据覆盖不够：(3,0) 从固定起点出发永远走不到。")


## 7. 贪心策略：漏学的状态给出错误策略

`π(s) = argmax_a Q(s,a)`。固定起点版在 `(3,0)` 上 Q 全是 0 → argmax 退化成"取第一个动作"（↑，直接掉进陷阱）。


In [ ]:
ARROW = {(0, 1): "↑", (0, -1): "↓", (-1, 0): "←", (1, 0): "→"}

def show_policy(Q, title):
    print(title)
    for y in range(H - 1, -1, -1):
        row = []
        for x in range(W):
            if (x, y) == GOAL:
                row.append("G")
            elif (x, y) == PIT:
                row.append("P")
            else:
                best = max(ACTS, key=lambda a: Q[((x, y), a)])
                row.append(ARROW[best])
        print("   ".join(row))
    print()

show_policy(Q, "固定起点版学到的策略（注意 (3,0) 是 ↑，主动往陷阱走）：")


## 8. 小结与下钻

- **DP 要模型、MC/TD 不要**；LLM 的 RL 只有采样一条路（DP 在语言里没有定义）。
- **TD 的关键是自举**：用"下一步的估计"替掉真实回报 → 有偏但方差小，且能在线。
- **Q-learning 的 `max` 带来最大化偏差**：修法是 Double Q，不是调 lr。
- **性能天花板常常不在算法，而在数据覆盖**：`(3,0)` 访问 0 次 → 完全没学到。LLM 里的对应物是 GRPO 的"组内全对/全错 = 这条 prompt 没有梯度信号"。

下一站：[环节 03 · 策略梯度与 Actor-Critic](./环节03-策略梯度与Actor-Critic详解.md)（不估分，直接调概率）。
